In [1]:
from langchain_community.vectorstores import Chroma
import os

from langchain_text_splitters import RecursiveCharacterTextSplitter
sample_text = """
Retrieval-Augmented Generation (RAG) is a technique that bridges the gap between 
a large language model's general knowledge and your specific, private data. 
By using local models like Llama 3.1 and local embeddings, you ensure that 
sensitive information never leaves your machine. The first step in any RAG 
pipeline is data ingestion, which involves loading documents and splitting 
them into chunks. Proper chunking is critical because if chunks are too small, 
they lose semantic meaning. If they are too large, they might exceed the 
model's context window or dilute the specific answer you are looking for.
""" * 10 

with open("local_rag_guide.txt", "w") as file:
    file.write(sample_text)

from langchain_community.document_loaders import TextLoader
loader = TextLoader("local_rag_guide.txt")
docs = loader.load()

print(f"Successfully loaded {len(docs)} document(s).")
print(f"Total character count: {len(docs[0].page_content)}")

small_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)
medium_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
large_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

small_chunks = small_splitter.split_documents(docs)
medium_chunks = medium_splitter.split_documents(docs)
large_chunks = large_splitter.split_documents(docs)

from langchain_community.embeddings import OllamaEmbeddings

ollama_embeddings = OllamaEmbeddings(model="nomic-embed-text")


persist_directory = "./chroma_db"

print("Creating vector database. This might take a moment...")

vector_db = Chroma.from_documents(
    documents=medium_chunks, 
    embedding=ollama_embeddings, 
    persist_directory=persist_directory 
)

/Users/pierson.chu/Downloads/github/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/pierson.chu/Downloads/github/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Successfully loaded 1 document(s).
Total character count: 6160
Creating vector database. This might take a moment...


/var/folders/gp/9snl51jj70dd01jy_yq1bcxr0000gp/T/ipykernel_30290/4191220527.py:36: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  ollama_embeddings = OllamaEmbeddings(model="nomic-embed-text")


**Now that we have our "filing cabinet" full of chunks and their corresponding mathematical vectors, we need to search it.**

When a user asks a question, we don't just do a standard keyword search (like hitting Ctrl+F). Instead, we take the user's question and pass it through the exact same embedding model we used for the chunks (in our case, nomic-embed-text). This turns the question into a 768-dimension vector.

Then, ChromaDB calculates the physical "distance" in that 768-dimensional space between the question's vector and every chunk's vector in the database. The chunks that are mathematically closest to the question are returned as the answers!

In [2]:
# 1. Define a user question (feel free to change this!)
query = "What happens if my chunks are too small or too large?"

# 2. Perform the similarity search
# We ask the database to return the top k=3 most relevant chunks, along with their scores
print(f"Searching for: '{query}'\n")
print("=" * 50)

# similarity_search_with_score returns a list of tuples: (Document, distance_score)
results = vector_db.similarity_search_with_score(query, k=3)

# 3. Display the results and their similarity scores
for i, (doc, score) in enumerate(results):
    print(f"--- MATCH #{i+1} ---")
    # Note: ChromaDB defaults to measuring "distance". So a LOWER score means it's a closer match!
    print(f"Distance Score: {score:.4f}") 
    print(f"Content: {doc.page_content}\n")

Searching for: 'What happens if my chunks are too small or too large?'

--- MATCH #1 ---
Distance Score: 333.0293
Content: they lose semantic meaning. If they are too large, they might exceed the 
model's context window or dilute the specific answer you are looking for.

--- MATCH #2 ---
Distance Score: 333.0293
Content: they lose semantic meaning. If they are too large, they might exceed the 
model's context window or dilute the specific answer you are looking for.

--- MATCH #3 ---
Distance Score: 333.0293
Content: they lose semantic meaning. If they are too large, they might exceed the 
model's context window or dilute the specific answer you are looking for.

